In [28]:
# For data manipulation
import pandas as pd
import numpy as np

# Machine learning libraries
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn import metrics
from sklearn.model_selection import cross_val_score

# To ignore unwanted warnings
import warnings
warnings.filterwarnings("ignore")

In [29]:
df = pd.read_csv("/Users/akash/PycharmProjects/Quant/Regression/market_data/GLD.csv")
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# Calculate the upward and downward deviations from the Open
df['Std_U'] = df['High']-df['Open']
df['Std_D'] = df['Open']-df['Low']

# Calculate 3-day moving average of close prices
df['S_3'] = df['Adj Close'].shift(1).rolling(window=3).mean()

# Calculate 15-day moving average of close prices
df['S_15'] = df['Adj Close'].shift(1).rolling(window=15).mean()

# Calculate 60-day moving average of close prices
df['S_60'] = df['Adj Close'].shift(1).rolling(window=60).mean()

# Calculate correlation between previous close and 3-day moving average for past 10 days
df['Corr'] = df['Adj Close'].shift(
    1).rolling(window=10).corr(df['S_3'].shift(1))

# Calculate OD, which shows changes since previous open
df['OD'] = df['Open']-df['Open'].shift(1)

# Calculate OL, which shows overnight changes
df['OL'] = df['Adj Close'].shift(1)-df['Open']

gold = df[['Adj Close', 'Open', 'Return','Std_U', 'Std_D', 'S_3', 'S_15', 'S_60', 'Corr', 'OD', 'OL']]

gold.dropna(inplace=True)
gold.isna().sum()

Adj Close    0
Open         0
Return       0
Std_U        0
Std_D        0
S_3          0
S_15         0
S_60         0
Corr         0
OD           0
OL           0
dtype: int64

In [30]:
# Initialise the Standard Scaler
scaler = StandardScaler()

# Scale the data in gold_prices and store it as an array in variable scaled
scaled = scaler.fit_transform(gold)

# Convert data stored in scaled from array to dataframe
scaled_gold_prices = pd.DataFrame(
    scaled, index=gold.index, columns=gold.columns)


# Independent variables
X = gold[['Open', 'S_3', 'S_15', 'S_60', 'OD', 'OL', 'Corr']]

# Dependent variable for upward deviation
yU = gold['Std_U']

# Dependent variable for downward deviation
yD = gold['Std_D']

# First we put scaling and then linear regression in the pipeline.
steps = [('scaler', StandardScaler()),
         ('linear', LinearRegression())]

# Define pipeline
pipeline = Pipeline(steps)

# Here we are using intercept as hyperparameter
parameters = {'linear__fit_intercept': [False, True]}

# We are using 80%-20% split, therefore splitting ratio will be 0.80
splitting_ratio = .80

# Split the data into two parts
# Use int to ensure that result is of integer data type.
split = int(splitting_ratio*len(gold))

# Define train dataset
X_train = X[:split]
yU_train = yU[:split]
yD_train = yD[:split]

# Define test data
X_test = X[split:]
yU_test = yU[split:]
yD_test = yD[split:]

# Use TimeSeriesSplit for cross validation
my_cv = TimeSeriesSplit(n_splits=5)

# Define reg as variable for GridSearch function containing pipeline, hyperparameters
reg = GridSearchCV(pipeline, parameters,
                   scoring='neg_mean_squared_error', cv=my_cv)

# Fit the model
reg.fit(X_train, yU_train)

# Initialise a best fit variable
best_fit = reg.best_params_

# Print best parameter
print(best_fit)

# Find the scores for 5 rounds of cross-validation
CV_scores = cross_val_score(reg, X_train, yU_train, cv=5,
                            scoring='neg_mean_squared_error')

# Print the scores for 5 rounds of cross-validation
print(CV_scores)

# Create a score variable
score = reg.best_score_

# Print the score
print(score)

{'linear__fit_intercept': True}
[-0.23173228 -0.14065577 -0.18767211 -0.60338892 -0.45169906]
-0.3752948798326371
